# ASC Dedup Validation — Renters (B-2961391)

Validation for the ASC dedup + termination changes ported into
`specialty-ltv/jobs/core/datalake_preprocess.py`.

Adapted from the home-ltv `compare_datapull_asc_dedup.ipynb`; paths and
column names changed to renters. Reads the deduped output (`asc_pol`) and
the pre-dedup snapshot (`pre_dup_asc_pol`) produced on branch feature/B-2961391.

**Run the Schema Check cell first.** If a column name differs from what's
referenced below, fix it before running the rest.

## Setup

In [ ]:
import ltv_helpers.non_spark_helpers as nsh
import ltv_helpers.pipeline_helpers as ph
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from specialty_ltv import paths as p
from ltv_helpers.spark import create_spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = create_spark(app_name="compare", buckets=p.buckets)

## Schema Check (run first)

Confirm the dedup key columns and output columns exist before running anything else.

In [ ]:
new = ph.read_parquet_s3(spark, p.asc_pol)            # deduped output
pre = ph.read_parquet_s3(spark, p.pre_dup_asc_pol)    # pre-dedup snapshot

print(p.asc_pol)
print(p.pre_dup_asc_pol)

new.printSchema()
pre.printSchema()

## Row Counts — Before vs After

Story point 3/4: pre-dedup vs deduped row counts, and how much dedup removed.

In [ ]:
print("pre-dedup :", pre.count())
print("deduped   :", new.count())
print("removed   :", pre.count() - new.count())

## Distinct Policy Count

In [ ]:
new.select(
    F.count("ply_policy_id"),
    F.countDistinct("ply_policy_id"),
).show()

## No Duplicate Policy + Version Rows

Story point 4. First line (pre-dedup, raw keys) should be > 0 — duplicates
existed. Second line (deduped output) should be 0 — dedup removed them.

In [ ]:
# pre-dedup: duplicates on the raw datalake keys
pre.groupBy("policyNbr", "policyVersionNbr").count().filter("count > 1").count()

In [ ]:
# deduped output: should be 0
new.groupBy("ply_policy_nbr", "drv_endorse_dt").count().filter("count > 1").count()

## Sample — Dedup Before / After

Two records showing the dedup. Picks one policy that had duplicate versions
in the pre-dedup file, shows the versions before and the single survivor after.

In [ ]:
# a policy that had duplicate versions
dup_pol = (
    pre.groupBy("policyNbr").count().filter("count > 1")
    .limit(1).collect()[0]["policyNbr"]
)
print("policy:", dup_pol)

before = pd.DataFrame(
    pre.filter(F.col("policyNbr") == dup_pol).take(100),
    columns=pre.columns,
)
before.sort_values(["policyVersionNbr"])   # the duplicates

In [ ]:
after = pd.DataFrame(
    new.filter(F.col("ply_policy_nbr") == dup_pol).take(100),
    columns=new.columns,
)
after   # the single survivor

## Sample — Termination Backfill

Added scope (termination indicator). Eyeball: `drv_termination_dt` is stamped
on all of a terminated policy's rows, and no endorsement is dated after it.

In [ ]:
term = pd.DataFrame(
    new.filter("drv_termination_dt is not null").take(10),
    columns=new.columns,
)
term.sort_values(["ply_policy_nbr", "drv_endorse_dt"])

## Not included (next week)

Needs an `ltv_base` job run off the new `asc_pol`, so deferred:

- LTV base old-vs-new comparison (`task_base` / `ltv_base.py`)
- PLUM training / delivery / internal results comparisons
- Home-only column diffs from the source notebook (`asc_llh_ind`, windhail,
  `flat_cancel_ind`, `max_ntr_plum`) — those columns do not exist renters-side.